# 🔀 Notebook 1: CQRS — Command Query Responsibility Segregation

**CQRS** stands for **Command Query Responsibility Segregation**. Fancy name, simple idea:

> Use **one model for writing** data, and a **different model for reading** it.

**Why?** Because reads and writes want *different* things:

| Writes (Commands)              | Reads (Queries)                  |
|--------------------------------|----------------------------------|
| Correctness, validation        | Speed, low latency               |
| Normalised tables, no dupes    | Denormalised, pre-joined         |
| Few per second                 | Thousands per second             |
| "Place this order"             | "Show my dashboard"              |

A real-world analogy: a **restaurant kitchen** (write side) prepares food carefully and tracks every ingredient. The **menu & table displays** (read side) are pre-formatted for guests to read quickly. You don't hand guests the inventory spreadsheet.

This notebook goes **bad → better → best**:

1. 😵 One shared model doing both jobs (pain).
2. 🙂 Split into a command side + a query projection.
3. ✅ Keep the projection in sync automatically.


## 🛠️ Setup

```bash
cd 05-microservices/cqrs
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## 😵 Step 1 — The "bad" way: one shared model

Let's model a tiny e-commerce system: **users** and **orders**.

We'll store them in plain Python dicts/lists (think of this as your SQL tables).
The **dashboard** query — *"for each user, show total spent and order count"* — has to
join and aggregate the raw tables every time it runs.

In [1]:
# Shared, normalised model - good for writes, painful for reads.
users = {}           # user_id -> {'name', 'email'}
orders = []          # [{'id', 'user_id', 'total'}, ...]

def create_user(uid, name, email):
    users[uid] = {'name': name, 'email': email}

def place_order(oid, uid, total):
    if uid not in users:
        raise ValueError('unknown user')
    orders.append({'id': oid, 'user_id': uid, 'total': total})

create_user(1, 'Ada',   'ada@x.io')
create_user(2, 'Grace', 'grace@x.io')
place_order(101, 1, 42.0)
place_order(102, 1, 10.0)
place_order(103, 2, 99.0)

# The dashboard query: recompute from scratch every time.
def dashboard_slow():
    result = {uid: {'name': u['name'], 'orders': 0, 'spent': 0.0}
              for uid, u in users.items()}
    for o in orders:                     # O(N) scan of every order
        r = result[o['user_id']]
        r['orders'] += 1
        r['spent']  += o['total']
    return result

dashboard_slow()


{1: {'name': 'Ada', 'orders': 2, 'spent': 52.0},
 2: {'name': 'Grace', 'orders': 1, 'spent': 99.0}}

### Why is that bad?

- Every dashboard request scans **all orders**. Fine for 3 rows, deadly for 3 million.
- Writes and reads fight over the same tables (locks, slow queries).
- To speed reads, you'd add indexes/caches that slow writes down. 🐢

Let's measure the pain with a bigger dataset.

In [2]:
import time, random

# Pretend we have 200k orders across 1000 users
users.clear(); orders.clear()
for uid in range(1, 1001):
    create_user(uid, f'user{uid}', f'u{uid}@x.io')
for oid in range(200_000):
    place_order(oid, random.randint(1, 1000), round(random.random()*100, 2))

t0 = time.perf_counter()
_ = dashboard_slow()
print(f'dashboard_slow over {len(orders):,} orders: {(time.perf_counter()-t0)*1000:.1f} ms')


dashboard_slow over 200,000 orders: 10.3 ms


## 🙂 Step 2 — Split: command side + query projection

**CQRS idea:** keep the normalised tables for writes, but also maintain a
**projection** — a pre-computed, read-optimised view.

- Command side answers: *"Is this write valid? Save it."*
- Query side answers: *"What should I show the user right now?"*

The projection lives in a different place (could be Redis, Elasticsearch,
a materialised SQL view). Here we simulate it with another dict.

In [3]:
# Query-side projection: denormalised, O(1) lookup per user.
user_summary = {}   # user_id -> {'name', 'orders', 'spent'}

def rebuild_projection():
    user_summary.clear()
    for uid, u in users.items():
        user_summary[uid] = {'name': u['name'], 'orders': 0, 'spent': 0.0}
    for o in orders:
        s = user_summary[o['user_id']]
        s['orders'] += 1
        s['spent']  += o['total']

t0 = time.perf_counter()
rebuild_projection()
print(f'projection built from {len(orders):,} orders in {(time.perf_counter()-t0)*1000:.1f} ms')

# Show a single user's dashboard - now just a dict lookup.
def dashboard_fast(uid):
    return user_summary[uid]

t0 = time.perf_counter()
for _ in range(10_000):
    dashboard_fast(42)
print(f'10,000 projection lookups: {(time.perf_counter()-t0)*1000:.1f} ms')
print('sample:', dashboard_fast(42))


projection built from 200,000 orders in 10.2 ms
10,000 projection lookups: 0.3 ms
sample: {'name': 'user42', 'orders': 232, 'spent': 11778.560000000003}


### Better — but still not great

We rebuild the entire projection from scratch. That's a **full re-scan** whenever
data changes. For a real system we want **incremental updates**: when a command
succeeds, we nudge the projection.

## ✅ Step 3 — Keep the projection in sync on every command

We route every write through a function that (1) updates the write model and
(2) updates the projection. In a real system the two sides live in different
databases and are connected by a message bus (Kafka, RabbitMQ, Postgres LISTEN,
etc.). The shape of the code is the same.

In [4]:
users.clear(); orders.clear(); user_summary.clear()

# --- Command handlers: the ONLY way to change state ---
def handle_create_user(uid, name, email):
    if uid in users:
        raise ValueError('user exists')
    users[uid] = {'name': name, 'email': email}
    # Update the projection incrementally.
    user_summary[uid] = {'name': name, 'orders': 0, 'spent': 0.0}

def handle_place_order(oid, uid, total):
    if uid not in users:
        raise ValueError('unknown user')
    orders.append({'id': oid, 'user_id': uid, 'total': total})
    s = user_summary[uid]
    s['orders'] += 1
    s['spent']  += total

# --- Query handler: only reads the projection ---
def query_dashboard(uid):
    return user_summary[uid]

handle_create_user(1, 'Ada', 'ada@x.io')
handle_place_order(101, 1, 42.0)
handle_place_order(102, 1, 10.0)

print(query_dashboard(1))


{'name': 'Ada', 'orders': 2, 'spent': 52.0}


### What did we gain?

- 🟢 **Reads are O(1)**, independent of write volume.
- 🟢 Read side can scale **independently** (e.g., Redis replicas).
- 🟢 Write side stays **clean and normalised**.
- 🟡 Trade-off: we wrote more code and have two places to keep consistent.

### What did we NOT solve yet?

- What if the projection update fails *after* the write succeeds? → **eventual consistency**.
- How do we add a **new** projection later, for a report we didn't think of? → **event sourcing** (Notebook 2).
- When should we *not* use CQRS? → Notebook 3.
